[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Polars*.

## ¿Qué es *Polars*?

[*Polars*](https://pola.rs/) es una biblioteca de *dataframes* escrita en *Rust*. Usa el formato columnar de *Apache Arrow* (estudiado en el *notebook* 34) como representación en memoria y reparte el trabajo entre todos los núcleos del procesador de forma automática.

Sus características principales son:

* **Sin índice de renglones.** Un *dataframe* de *Polars* es sólo un conjunto de columnas; los renglones se identifican por su posición o por el valor de alguna columna.
* **Expresiones.** Las operaciones se describen con expresiones (`pl.col('precio') * 1.16`) que se evalúan dentro de un contexto (`select()`, `with_columns()`, `filter()`, `group_by().agg()`). Esto permite a *Polars* optimizar y paralelizar el cálculo.
* **Tipos estrictos.** Cada columna tiene un tipo fijo y los nulos se representan igual para cualquier tipo.
* **Ejecución inmediata o diferida.** Además del modo inmediato (*eager*), que se usa en este *notebook*, *Polars* tiene un modo diferido (*lazy*) que optimiza una consulta completa antes de ejecutarla. Se estudia en el *notebook* 37.

### *Pandas* 3 y *Polars*.

| Aspecto | *Pandas* 3 | *Polars* |
| :--- | :--- | :--- |
| Implementación | *Python*, *C* y *Cython* sobre *NumPy* | *Rust* |
| Formato en memoria | *NumPy*; *Arrow* para texto y, opcionalmente, para otras columnas | *Arrow* |
| Índice de renglones | Sí | No |
| Paralelismo | La mayoría de las operaciones usan un solo hilo | Multihilo automático |
| Modo de ejecución | Inmediato | Inmediato (*eager*) o diferido (*lazy*) |
| Estilo de código | Métodos e indexación con corchetes | Expresiones dentro de contextos |
| Datos más grandes que la memoria | No | En muchos casos, con el motor *streaming* |
| Ecosistema | Muy amplio y maduro | Más reciente; se integra con otras herramientas mediante *Arrow* |

*Polars* suele ser más rápido que *Pandas*, sobre todo con datos grandes y en operaciones de agrupación y combinación de tablas. El *notebook* 37 incluye una medición. *Pandas* sigue siendo la opción con más integraciones (por ejemplo, con *scikit-learn*, *statsmodels* o *Seaborn*), y la conversión entre ambas bibliotecas es sencilla.

[Guía de usuario de *Polars*](https://docs.pola.rs/user-guide/) · [Referencia de la *API*](https://docs.pola.rs/api/python/stable/reference/index.html)

* La siguiente celda instala *Polars* en el entorno actual en caso de que no esté disponible.

In [ ]:
%pip install polars

* La siguiente celda importa `polars` con el alias convencional `pl` y el módulo de selectores `polars.selectors` con el alias `cs`, que se explica más adelante. `pl.Config.set_fmt_float('full')` hace que los números de punto flotante se muestren completos en lugar de en notación científica.

In [ ]:
import os

import polars as pl
import polars.selectors as cs

pl.Config.set_fmt_float('full')
print(f"Polars {pl.__version__}")

* La siguiente celda define las rutas de los datos de ejemplo, descritos en el *notebook* 34, y crea el directorio `data_tmp` para los archivos que se generen.

In [ ]:
ruta_ventas = os.path.join('data', 'ventas.csv')
ruta_tiendas = os.path.join('data', 'tiendas.csv')
dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)

## *Dataframes* y series.

Un *dataframe* de *Polars* (`pl.DataFrame`) es una tabla formada por columnas con nombre; cada columna es una serie (`pl.Series`) de un único tipo.

```
pl.DataFrame(<datos>, schema=<schema>)
```

Donde:

* `<datos>` es un diccionario de listas, una lista de diccionarios, un arreglo de *NumPy* o un *dataframe* de *Pandas*, entre otros.
* `<schema>` es opcional: un diccionario `{nombre: tipo}` con los tipos de las columnas. Si se omite, *Polars* infiere los tipos.

| Atributo o método | Descripción |
| :--- | :--- |
| [`df.shape`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html) | Tupla con el número de renglones y de columnas. |
| [`df.height`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.height.html) / [`df.width`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.width.html) | Número de renglones / de columnas. |
| [`df.columns`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.columns.html) | Lista con los nombres de las columnas. |
| [`df.schema`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html) | Nombres y tipos de las columnas. |
| [`df.head(n)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html) / [`df.tail(n)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.tail.html) | Primeros / últimos `n` renglones (5 por omisión). |
| [`df.glimpse()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.glimpse.html) | Vista compacta: una línea por columna con su tipo y primeros valores. |
| [`df.describe()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.describe.html) | Estadísticas descriptivas de cada columna. |

**Ejemplos:**

* La siguiente celda crea un *dataframe* pequeño a partir de un diccionario de listas.

In [ ]:
productos = pl.DataFrame({
    'producto': ['Cuaderno profesional', 'Mochila escolar', 'Balón de fútbol', 'Cafetera de goteo'],
    'categoria': ['Papelería', 'Papelería', 'Deportes', 'Hogar'],
    'precio': [65.0, 649.0, 459.0, 799.0],
    'existencias': [120, 35, 0, 18],
})
productos

* La siguiente celda muestra el *schema*. Los tipos de *Polars* se escriben con mayúscula inicial: `String`, `Float64`, `Int64`.

In [ ]:
productos.schema

* La siguiente celda crea el mismo *dataframe* indicando tipos más pequeños para `precio` y `existencias`.

In [ ]:
pl.DataFrame(
    productos.to_dict(),
    schema={'producto': pl.String, 'categoria': pl.String, 'precio': pl.Float32, 'existencias': pl.Int16},
).schema

* La siguiente celda obtiene la columna `precio` con corchetes. El resultado es una serie, que tiene nombre y tipo pero no índice.

In [ ]:
precios = productos['precio']
precios

* Como no hay índice, los renglones se obtienen por posición. La siguiente celda obtiene el segundo renglón como tupla con `row()` y los dos primeros como *dataframe* con `slice()`.

In [ ]:
print(productos.row(1))
productos.slice(0, 2)

## Lectura y escritura de archivos.

| Formato | Lectura | Escritura |
| :--- | :--- | :--- |
| *CSV* | [`pl.read_csv()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html) | [`df.write_csv()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_csv.html) |
| *Parquet* | [`pl.read_parquet()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_parquet.html) | [`df.write_parquet()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_parquet.html) |
| *Feather* / *Arrow IPC* | [`pl.read_ipc()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_ipc.html) | [`df.write_ipc()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_ipc.html) |
| *JSON* | [`pl.read_json()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_json.html) | [`df.write_json()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_json.html) |
| *Excel* | [`pl.read_excel()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_excel.html) | [`df.write_excel()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_excel.html) |
| Base de datos | [`pl.read_database()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_database.html) | [`df.write_database()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_database.html) |

Cada función `read_*` tiene una equivalente `scan_*` (por ejemplo, `pl.scan_parquet()`) para el modo diferido, que se estudia en el *notebook* 37.

Algunos argumentos útiles de `pl.read_csv()` son:

* `try_parse_dates=True` convierte a fecha las columnas de texto con formato de fecha.
* `schema_overrides={columna: tipo}` fija el tipo de algunas columnas; las demás se infieren.
* `columns=[...]` lee sólo las columnas indicadas.

**Ejemplos:**

* La siguiente celda lee `data/ventas.csv`, convierte las fechas y fija el tipo de `tienda_id`.

In [ ]:
ventas = pl.read_csv(ruta_ventas, try_parse_dates=True, schema_overrides={'tienda_id': pl.Int16})
ventas.shape

* La siguiente celda muestra una vista compacta del *dataframe* con `glimpse()`.

In [ ]:
ventas.glimpse()

* La siguiente celda calcula estadísticas descriptivas de las columnas numéricas con `describe()`.

In [ ]:
ventas.select('cantidad', 'precio_unitario', 'descuento').describe()

* La siguiente celda escribe el *dataframe* en formato *Parquet* y lo lee de nuevo. A diferencia del *CSV*, el archivo *Parquet* conserva los tipos de las columnas.

In [ ]:
ruta_parquet = os.path.join(dir_tmp, 'ventas_polars.parquet')
ventas.write_parquet(ruta_parquet)

pl.read_parquet(ruta_parquet).schema == ventas.schema

## Contextos y expresiones.

En *Polars*, la mayoría de las operaciones se escriben con **expresiones**. Una expresión describe un cálculo sobre una o más columnas, por ejemplo `pl.col('precio') * 1.16`, pero no hace nada por sí sola: se evalúa cuando se usa dentro de un **contexto**.

| Contexto | Resultado |
| :--- | :--- |
| [`df.select(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.select.html) | Un *dataframe* sólo con las columnas que producen las expresiones. |
| [`df.with_columns(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html) | El *dataframe* original con columnas agregadas o reemplazadas. |
| [`df.filter(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html) | Los renglones en los que las expresiones booleanas son verdaderas. |
| [`df.group_by(<claves>).agg(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by.html) | Un renglón por grupo, con una columna por expresión. |

Todos los contextos regresan un *dataframe* nuevo; el original no se modifica. Dentro de un contexto, *Polars* puede evaluar varias expresiones en paralelo.

[`pl.col(<nombre>)`](https://docs.pola.rs/api/python/stable/reference/expressions/col.html) es la expresión más básica: hace referencia a una columna. El *notebook* 36 estudia las expresiones a fondo.

* La siguiente celda muestra que una expresión fuera de un contexto es sólo una descripción del cálculo.

In [ ]:
expresion = pl.col('precio_unitario') * 1.16
expresion

* La siguiente celda evalúa la misma expresión dentro de `select()`. Por omisión, la columna resultante conserva el nombre de la primera columna de la expresión; `alias()` le asigna otro nombre.

In [ ]:
ventas.select(
    'producto',
    pl.col('precio_unitario'),
    expresion.round(2).alias('precio_con_iva'),
).head(3)

## Selección de columnas.

Dentro de `select()` (y de los demás contextos) las columnas pueden indicarse de varias formas:

| Forma | Selecciona |
| :--- | :--- |
| `'producto'` o `pl.col('producto')` | Una columna por nombre. |
| `pl.col('producto', 'cantidad')` | Varias columnas por nombre. |
| `pl.col('^cli.*$')` | Las columnas cuyo nombre coincide con la expresión regular. **El patrón debe empezar con `^` y terminar con `$`**; de lo contrario se interpreta como un nombre literal. |
| `pl.col(pl.Float64)` | Las columnas de un tipo. |
| `pl.all()` | Todas las columnas. |
| `pl.exclude('venta_id', 'cliente_email')` | Todas, excepto las indicadas. |

### Selectores.

El módulo [`polars.selectors`](https://docs.pola.rs/api/python/stable/reference/selectors.html), que se importa como `cs`, ofrece funciones para seleccionar columnas por tipo o por nombre. Los selectores se pueden combinar con operaciones de conjuntos: `|` (unión), `&` (intersección) y `-` (diferencia).

| Selector | Selecciona |
| :--- | :--- |
| `cs.numeric()` | Columnas numéricas. |
| `cs.string()` | Columnas de texto. |
| `cs.temporal()` | Columnas de fecha y hora. |
| `cs.starts_with('...')` / `cs.ends_with('...')` / `cs.contains('...')` | Columnas cuyo nombre empieza con, termina con o contiene un texto. |
| `cs.by_name(...)` / `cs.by_dtype(...)` | Columnas por nombre o por tipo. |

**Ejemplos:**

* La siguiente celda selecciona dos columnas por nombre.

In [ ]:
ventas.select('producto', 'cantidad').head(3)

* La siguiente celda selecciona con una expresión regular las columnas cuyo nombre termina en `_id`.

In [ ]:
ventas.select(pl.col('^.*_id$')).head(3)

* La siguiente celda intenta usar un patrón sin `^` y `$`. *Polars* lo interpreta como el nombre literal de una columna y desencadena una excepción de tipo `ColumnNotFoundError`.

In [ ]:
ventas.select(pl.col('.*_id'))

* La siguiente celda selecciona todas las columnas excepto `venta_id` y `cliente_email`.

In [ ]:
ventas.select(pl.exclude('venta_id', 'cliente_email')).head(3)

* La siguiente celda usa selectores para obtener las columnas numéricas, excepto los identificadores.

In [ ]:
ventas.select(cs.numeric() - cs.ends_with('_id')).head(3)

* La siguiente celda combina un selector de texto y uno de fecha con `|`.

In [ ]:
ventas.select(cs.string() | cs.temporal()).head(3)

## Filtros.

[`df.filter()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html) conserva los renglones en los que una expresión booleana es verdadera.

```
df.filter(<condición_1>, <condición_2>, ...)
```

Donde cada `<condición>` es una expresión booleana. Si se indican varias, deben cumplirse todas.

| Operación | Expresión |
| :--- | :--- |
| Comparación | `pl.col('cantidad') > 3`, `pl.col('canal') == 'en línea'` |
| Y, o, negación | `(<cond_1>) & (<cond_2>)`, `(<cond_1>) \| (<cond_2>)`, `~(<cond>)` |
| Pertenencia a una lista | `pl.col('categoria').is_in(['Hogar', 'Deportes'])` |
| Rango (incluye los extremos) | `pl.col('precio_unitario').is_between(100, 500)` |
| Nulos | `pl.col('descuento').is_null()`, `pl.col('descuento').is_not_null()` |

Los paréntesis son necesarios al combinar condiciones con `&`, `|` y `~`, porque estos operadores tienen mayor precedencia que las comparaciones.

**Ejemplos:**

* La siguiente celda obtiene las ventas en línea de más de 3 piezas.

In [ ]:
ventas.filter(pl.col('canal') == 'en línea', pl.col('cantidad') > 3).select(
    'fecha_hora', 'canal', 'producto', 'cantidad'
).head(3)

* La siguiente celda obtiene las ventas de `Hogar` o `Deportes` con precio unitario entre 300 y 500 pesos.

In [ ]:
ventas.filter(
    pl.col('categoria').is_in(['Hogar', 'Deportes']),
    pl.col('precio_unitario').is_between(300, 500),
).select('producto', 'categoria', 'precio_unitario').head(5)

* La siguiente celda obtiene las ventas de diciembre de 2025 o de más de 4 piezas, usando `|`.

In [ ]:
ventas.filter(
    (pl.col('fecha_hora') >= pl.datetime(2025, 12, 1)) | (pl.col('cantidad') > 4)
).height

## Transformaciones.

| Método | Descripción |
| :--- | :--- |
| [`df.with_columns()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html) | Agrega columnas o reemplaza las existentes. |
| [`df.rename()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.rename.html) | Renombra columnas con un diccionario `{actual: nuevo}`. |
| [`df.drop()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop.html) | Elimina columnas. |
| [`df.sort()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html) | Ordena por una o varias columnas. |
| [`df.unique()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unique.html) | Elimina renglones duplicados, considerando todas las columnas o sólo las indicadas en `subset`. |
| [`df.top_k()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.top_k.html) | Los `k` renglones con los valores más altos de una columna, sin ordenar todo el *dataframe*. |

### El método `with_columns()`.

```
df.with_columns(
    <expresión_1>.alias('<nombre_1>'),
    <nombre_2>=<expresión_2>,
    ...
)
```

Donde cada columna nueva se nombra con `alias()` o con un argumento por nombre. Si el nombre coincide con una columna existente, ésta se reemplaza. Las expresiones de una misma llamada se calculan en paralelo, por lo que no pueden usar una columna creada en la misma llamada; para eso se encadena otro `with_columns()`.

**Ejemplos:**

* La siguiente celda agrega el importe bruto de cada venta y el importe neto, que descuenta el porcentaje de descuento. Como `descuento` tiene nulos, éstos se tratan como `0` con `fill_null()`. El importe neto usa el importe bruto, por lo que se calcula en un segundo `with_columns()`.

In [ ]:
ventas = (
    ventas
    .with_columns(importe=pl.col('cantidad') * pl.col('precio_unitario'))
    .with_columns(
        (pl.col('importe') * (1 - pl.col('descuento').fill_null(0))).round(2).alias('importe_neto')
    )
)
ventas.select('cantidad', 'precio_unitario', 'descuento', 'importe', 'importe_neto').head(5)

* La siguiente celda renombra dos columnas y elimina otra. El *dataframe* `ventas` no cambia porque el resultado no se asigna.

In [ ]:
ventas.rename({'precio_unitario': 'precio', 'cliente_email': 'cliente'}).drop('venta_id').columns

* La siguiente celda ordena por categoría (ascendente) e importe neto (descendente).

In [ ]:
ventas.sort(['categoria', 'importe_neto'], descending=[False, True]).select(
    'categoria', 'producto', 'importe_neto'
).head(5)

* La siguiente celda obtiene las tres ventas con mayor importe neto con `top_k()`.

In [ ]:
ventas.top_k(3, by='importe_neto').select('fecha_hora', 'producto', 'cantidad', 'importe_neto')

* La siguiente celda obtiene las combinaciones únicas de categoría y producto, ordenadas.

In [ ]:
ventas.select('categoria', 'producto').unique().sort('categoria', 'producto')

## Valores nulos.

*Polars* representa los valores faltantes como `null` en cualquier tipo de dato. `NaN` (*not a number*) es distinto: es un valor de punto flotante válido que resulta de operaciones como `0 / 0`, y no se considera nulo.

| Método o expresión | Descripción |
| :--- | :--- |
| [`df.null_count()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html) | Número de nulos por columna. |
| `pl.col(...).is_null()` / `is_not_null()` | Expresión booleana para filtrar. |
| [`pl.col(...).fill_null(<valor>)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.fill_null.html) | Reemplaza los nulos por un valor o con una estrategia: `strategy='forward'`, `'backward'`, `'mean'`, `'min'`, `'max'` o `'zero'`. |
| [`df.drop_nulls(subset=...)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop_nulls.html) | Elimina los renglones con nulos en las columnas indicadas. |

**Ejemplos:**

* La siguiente celda cuenta los nulos de cada columna. Sólo `descuento` tiene nulos.

In [ ]:
ventas.null_count()

* La siguiente celda compara el promedio del descuento ignorando los nulos (comportamiento por omisión) con el que resulta de tratarlos como `0`.

In [ ]:
ventas.select(
    pl.col('descuento').mean().alias('ignorando_nulos'),
    pl.col('descuento').fill_null(0).mean().alias('nulos_como_cero'),
)

* La siguiente celda elimina los renglones sin descuento y muestra cuántos quedan.

In [ ]:
ventas.drop_nulls(subset='descuento').height

* La siguiente celda muestra la diferencia entre `NaN` y nulo: sólo el valor `None` cuenta como nulo.

In [ ]:
pl.DataFrame({'x': [1.0, None, float('nan')]}).with_columns(
    es_nulo=pl.col('x').is_null(),
    es_nan=pl.col('x').is_nan(),
)

## Agrupación y agregación.

```
df.group_by(<claves>, maintain_order=<booleano>).agg(<expresiones>)
```

Donde:

* `<claves>` son una o varias columnas (o expresiones) que definen los grupos.
* `maintain_order=True` conserva los grupos en el orden en que aparecen por primera vez. Por omisión el orden no está garantizado, porque los grupos se calculan en paralelo.
* `<expresiones>` son las agregaciones; cada una produce una columna del resultado.

| Expresión | Resultado por grupo |
| :--- | :--- |
| `pl.len()` | Número de renglones. |
| `pl.col(...).sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.std()` | Estadísticas de la columna. |
| `pl.col(...).n_unique()` | Número de valores distintos. |
| `pl.col(...).first()`, `.last()` | Primer o último valor. |

El objeto agrupado también tiene atajos como `len()`, `sum()` o `mean()`, que aplican la agregación a todas las columnas que no son claves.

[Documentación de `group_by()`](https://docs.pola.rs/api/python/stable/reference/dataframe/group_by.html)

**Ejemplos:**

* La siguiente celda calcula, por categoría, el número de ventas, las piezas vendidas, el importe neto total y el número de productos distintos; después ordena por importe.

In [ ]:
(
    ventas
    .group_by('categoria')
    .agg(
        pl.len().alias('ventas'),
        pl.col('cantidad').sum().alias('piezas'),
        pl.col('importe_neto').sum().round(2).alias('importe_neto'),
        pl.col('producto').n_unique().alias('productos'),
    )
    .sort('importe_neto', descending=True)
)

* La siguiente celda agrupa por dos claves, categoría y canal, y calcula el ticket promedio.

In [ ]:
(
    ventas
    .group_by('categoria', 'canal')
    .agg(pl.col('importe_neto').mean().round(2).alias('ticket_promedio'))
    .sort('categoria', 'canal')
)

* La siguiente celda usa como clave una expresión: el año de la venta, que se extrae de `fecha_hora` con `dt.year()`.

In [ ]:
ventas.group_by(pl.col('fecha_hora').dt.year().alias('anio')).agg(
    pl.col('importe_neto').sum().round(2)
).sort('anio')

* Un objeto agrupado también se puede recorrer con un ciclo `for`. Cada elemento es una tupla con la clave y el *dataframe* del grupo. La clave es a su vez una tupla, con un valor por cada columna de agrupación.

In [ ]:
for (canal,), grupo in ventas.group_by('canal', maintain_order=True):
    print(f"{canal:10s} {grupo.height:6,} ventas")

## Conversión a otros formatos.

| Función o método | Descripción |
| :--- | :--- |
| [`df.to_pandas()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_pandas.html) | Convierte a un *dataframe* de *Pandas*. Hace una copia, salvo con `use_pyarrow_extension_array=True`. |
| [`pl.from_pandas(<df>)`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_pandas.html) | Crea un *dataframe* de *Polars* a partir de uno de *Pandas*. |
| [`df.to_numpy()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_numpy.html) | Convierte a un arreglo de *NumPy* de dos dimensiones. |
| [`df.to_arrow()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_arrow.html) / [`pl.from_arrow()`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_arrow.html) | Convierte hacia y desde una *table* de *PyArrow* (ver el *notebook* 34). |
| [`df.to_dicts()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_dicts.html) | Lista de diccionarios, uno por renglón. |

Una práctica común es hacer la carga y las transformaciones con *Polars* y convertir a *Pandas* sólo al final, cuando una biblioteca lo requiere.

**Ejemplos:**

* La siguiente celda convierte el resumen por categoría a *Pandas*. El texto se convierte al tipo `str` de *Pandas* 3.

In [ ]:
resumen = ventas.group_by('categoria').agg(pl.col('importe_neto').sum().round(2)).sort('categoria')
resumen_pd = resumen.to_pandas()
print(type(resumen_pd))
resumen_pd.dtypes

* La siguiente celda convierte el *dataframe* de *Pandas* de vuelta a *Polars*. El índice de *Pandas* se descarta.

In [ ]:
pl.from_pandas(resumen_pd)

* La siguiente celda convierte dos columnas numéricas a un arreglo de *NumPy*.

In [ ]:
ventas.select('cantidad', 'precio_unitario').head(3).to_numpy()

* La siguiente celda convierte los dos primeros renglones del catálogo de productos a una lista de diccionarios.

In [ ]:
productos.head(2).to_dicts()

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| ***Polars*** | Biblioteca de *dataframes* en *Rust*, columnar (*Arrow*), multihilo y sin índice de renglones. |
| ***Dataframe* y serie** | Tabla de columnas con tipo fijo; los renglones se obtienen por posición. |
| **Lectura y escritura** | `read_*` y `write_*` para *CSV*, *Parquet*, *Feather*, *JSON*, *Excel* y bases de datos. |
| **Contextos** | `select()`, `with_columns()`, `filter()` y `group_by().agg()` evalúan expresiones y regresan un *dataframe* nuevo. |
| **Selección de columnas** | Por nombre, expresión regular (`^...$`), tipo o selectores (`cs`). |
| **Nulos** | `null` en cualquier tipo, distinto de `NaN`; `fill_null()`, `drop_nulls()`, `null_count()`. |
| **Agrupación** | `group_by(...).agg(...)` con una o varias claves; el orden de los grupos no está garantizado. |
| **Conversiones** | `to_pandas()`, `from_pandas()`, `to_numpy()`, `to_arrow()`. |

El *notebook* 36 estudia las expresiones a fondo, y el 37 el modo diferido, la combinación de tablas y otras técnicas avanzadas.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>